
## Objectifs
##### 1 Lire et traiter des données d'entrée avec [PANDAS](https://pandas.pydata.org/)

##### 2 Représenter la distribution empirique des "fold-change" (**FC**) vs Précision, aka: volcano-plot
    
##### 3 Effectuer une analyse de sur-représentation en termes GO


## Ressources : Rappels et illustration de l'analyse de représentation de termes GO
###### [TP de MADP](https://github.com/glaunay/tp-proteomics#3-obtention-des-param%C3%A8tres-du-mod%C3%A8le)
###### [Fiche Bioconductor](https://www.bioconductor.org/help/course-materials/2015/SeattleApr2015/E_GeneSetEnrichment.html)



#### Fichiers necessaires:
* `TCL_wt1.tsv`: l'experience de protéomique                       
* `tree_ecoli_K12_measured_redis.pik`: Un arbre d'ontologie GO (voir plus bas)
* `uniprot_store_ecoli_K12.pik`: le protéome uniprot d'E.Coli


In [3]:
import pandas
import numpy as np

In [4]:
%matplotlib inline
import matplotlib.pyplot as plt

### Lecture des valeurs experimentales
Charger ```TCL_wt1.tsv``` dans une [dataframe pandas](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html).
<br/><span style="color:firebrick;font-weight:bold"> ▶ 
Retirer les lignes présentant des aberrations numériques
</span>
<br/><span style="color:firebrick;font-weight:bold">▶ Attention aux types des colonnes !</span>

##### 1. Un premier volcano-plot

Ce type de figure représente pour chaque protéine en abscisse le taux d'abondance et en ordonnée la précision de la mesure.

<span style="color:firebrick;font-weight:bold">▶ 1. Réaliser un scatter plot matplotlib avec</span>
<br/><span style="color:firebrick;font-weight:bold">- en x, `Log2 Corrected Abundance Ratio`</span>
<br/><span style="color:firebrick;font-weight:bold">-  en y, `LOG10 Adj.P-val`</span>

<span style="color:firebrick;font-weight:bold">▶Vous légenderez les axes, attention `LOG10 Adj.P-val` est en fait `-LOG10 Adj.P-val`, regardez bien.</span>

##### 2. Réaliser des versions évoluées du graphique précédent
L'objectif est de mettre l'accent sur la région **-Log10 adj Pvalue > 4** et **Log2 corrected abundance ratio > 0**.
<br/>Pour cela vous revisiterez le scatter plot précedent au travers des versions suivantes:
<br/><span style="color:firebrick;font-weight:bold"> ▶ première version : un rectangle de la couleur de votre choix matérialisera la région</span>
<br/><span style="color:firebrick;font-weight:bold"> ▶ deuxième version  : une couleur de votre choix représentera les points de la région</span>
<br/><span style="color:firebrick;font-weight:bold"> ▶ troisème version (optionnel): Les identifiants uniprot remplaceront les points des protéines de la région</span>

#### Analyse ORA

##### Principes
Vous disposez d'un arbre de termes de la [GeneOntology](https://www.geneontology.org/) dans lequel toutes les protéines detectées dans l'experience ontété renseignée. Cet arbre va vous permttre de calculerla sur-représentation (**ORA**) des termes GO portés par un groupe de protéine d'intérêt versus un groupe de protéines de référence. Le groupe de référence peut être tout le protéome de l'organisme ou uniquement les protéines détéctées dans l'experience.
**Vous devrez construire la liste de protéines interessantes**

L'arbre des termes GO est à créer à partir du ficher pickle nommé *tree_ecoli_K12_measured_redis.pik*
```python
import pickle
go_tree = pickle.load( open("PATH/TO/tree_ecoli_K12_measured_redis.pik", "rb" ) )
```
#### Cet objet a différente fonctionalités comme :
##### Regarder un terme GO, par son nom commun ou son identifiant GO :
```python
print(go_tree.view_go_node('metal ion transport'))
print(go_tree.view_go_node('GO:0030001'))
```
##### Lister les protéines qui portent un terme GO particulier. Attention, il peut y en avoir beaucoup!
```python
print(go_tree.get_proteins('GO:0030001'))
print(go_tree.get_proteins('metal ion transport'))
```
##### Calculer de scores et pvaleurs d'ORA à l'aide du module `obogo.statistics` sur:
Dans les exemples suivants, on considera comme protéines d'intérêt la liste factice suivante d'identifiants uniprot  : `['P02930', 'P03819', 'P0A910']`
###### => Un terme GO en particulier : 
La fonction `compute_node_ora`, attend trois arguments positonnels: l'arbre des termes GO et la liste de protéines d'intérêt et le terme GO à analyser (sous forme de nom ou d'identifiant).

Le calcul est, par défaut, réalisé en se basant sur la représentation du terme GO dans le tout le protéome.

La population, plus réduite, des protéines de l'experience peut être utilisée à la place, à l'aide du paramètre `norm` mis à la valeur *"measured"*.

```python
from obogo.statistics import compute_node_ora
print( compute_node_ora(go_tree, ['P02930', 'P03819', 'P0A910'], 'GO:0006811') )
print( compute_node_ora(go_tree, ['P02930', 'P03819', 'P0A910'], 'metal ion transport') )
print( compute_node_ora(go_tree, ['P02930', 'P03819', 'P0A910'], 'GO:0006811', norm='measured') )
```

###### => Tous les termes GO de l'arbre
La fonction `score_ora_tree` retournera un génerateur des scores de chaque terme GO. Le paramètre `norm` est aussi disponible.

```python
from obogo.statistics import score_ora_tree
for go_score in score_ora_tree(go_tree, ['P02930', 'P03819', 'P0A910']):
    print(go_score)
```

<span style="color:green;font-weight:bold">↑ Reproduire cet exemple dans la cellule ci-dessous.↓ </span>
</br><span style="color:firebrick;font-weight:bold"> ▶ Inspecter les objets retournés par la fonction  compute_node_ora (et les itérations de score_ora_tree), que voyez-vous ?</span>

###### 2/ Analyser la représentation des termes GO parmi les protéines surabondantes de l'experience
<span style="color:firebrick;font-weight:bold"> ▶ Vous devrez extraire les numéro d'accession uniprot des protéines identifées dans le scatter plot précedent.</span>
<br/><span style="color:firebrick;font-weight:bold"> ▶ Vous conduirez sur cette liste de protéines une analyse de sur-représentation en termes GO à l'aide du package obogo.</span>
<br/><span style="color:firebrick;font-weight:bold"> ▶ Vous ferez varier le seuil d'abondance, pour tester différentes compositions de protéines surabondantes.</span>
<br/><span style="color:firebrick;font-weight:bold"> ▶ Vous supprimerez les scores ORA qui vous semblent non pertinents.</span>
<br/><span style="color:firebrick;font-weight:bold"> ▶ Vous ordonnerez les résutats par significativité décroissante.</span>
<br/><span style="color:firebrick;font-weight:bold"> ▶ Vous séparerez les scores selon les namespaces de Gene Ontology auquels ils appartiennent (*biological_process, molecular_function ou cellular_component*).</span>



Vous pourrez stocker les résultats ainsi mis en forme au [les sauvegarder au format json](https://docs.python.org/3/library/json.html).


## Une première application: représentation riche du résultat de l'analyse


### Mise en forme "riche" des résultats grâce au notebook

Jupyter permet d'étendre le concept de la méthode **__str__()** afin de produire du contenu HTML.
 Ainsi, tout objet impémentant une méthode **_repr_html_()** sera affiché, via son rendu HTML, dans la cellule d'un notebook.

```python
class Point():
    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.z = z
    def _repr_html_(self):
        htmlString = "<table><thead><tr><td>Abscisse</td><td>Ordonnée</td><td>Cote</td></tr></thead>"
        htmlString += f"<tbody><tr><td>{self.x}</td><td>{self.y}</td><td>{self.z}</td></tr></tbody></table>"
        return htmlString
Point(2,3,4)
```
Affichera

![caption](figs/reprhtml.png)


### Intégration des résultats à la visualisation du notebook 

En vous inspirant de la fonction ci-dessus, implémenter une classe **Rich_GO_ORA** dont la méthode [`_repr_html_`](https://ipython.readthedocs.io/en/stable/config/integrating.html) permettra un affichage plus lisible d'une des analyses `biological_process / molecular_function /cellular_component`

Un exemple possible est,
![caption](figs/_repr_html_.png)

(Bonus): clicker sur le nom du terme GO envoie vers sa description.
Pour cela, afficher le nom du pathway dans une balise hyperlien pour permettre d'acceder à la page de description du terme GO. 

Les termes GO sont accessibles aux URL de la forme
`http://amigo.geneontology.org/amigo/term/GO:GO_NUMBER`.




## Une deuxième application: volcano plot améliorés
#### Combinons maintenant les données ORA et celles de table experimentale
##### Pour la catégorie de termes GO (biological_process / molecular_function /cellular_component) de votre choix
<span style="color:firebrick;font-weight:bold"> ▶ Selectionnez 4 termes GO que vous estimez les plus pertinents </span>
<br/><span style="color:firebrick;font-weight:bold"> ▶ Générer une grille de 4 volcano-plot, un pour chaque terme </span>
<br/><span style="color:firebrick;font-weight:bold"> ▶ Dans chaque volcano plot colorez dans une couleur différente les protéines porteuses du terme GO considéré </span>
<br/><span style="color:firebrick;font-weight:bold"> ▶ Donnez le nom de la catégorie générale (namespace GO) à la grille</span>
<br/><span style="color:firebrick;font-weight:bold"> ▶ Donnez le nom du terme GO représenté dans chaque graphique avec une couleur de titre cohérente </span>